## Communication Topology and Belief Dynamics in Multi-Agent LLM Reasoning
### Experiment Analysis & Visualisation

This notebook analyses experimental results from a multi-agent LLM system tested across different communication topologies on the GSM8K benchmark.

**Topologies tested:**
- **Independent**: agents reason alone, answers aggregated via majority vote
- **Fully Connected**: all agents see each other's responses before revising
- **Mediator**: a mediator summarises responses; agents see only the summary
- **Chain**: agents answer sequentially, each seeing only the previous agent

**Primary questions:**
1. Does collaboration improve accuracy over independent reasoning?
2. How do different communication structures affect convergence?
3. What are the cost/accuracy trade-offs across topologies?

In [203]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

import pandas as pd
import numpy as np
import warnings
import glob
from pathlib import Path
warnings.filterwarnings("ignore")

---
### 1. Setup & Data Loading

In [204]:
pio.templates.default = "plotly_white"

TOPO_COLORS = {
    "independent": "#636EFA",
    "full": "#EF553B",
    "mediator": "#00CC96",
    "chain": "#AB63FA",
}

AGENT_COLORS = {
    "gemma3:4b": "#FF6B6B",
    "phi4-mini": "#4ECDC4",
    "llama3.2:3b": "#45D15A",
    "qwen2.5:3b-instruct": "#F7DC6F",
}

AGENT_NAME_MAP = {
    0: "gemma3:4b", 
    1: "phi4-mini", 
    2: "llama3.2:3b", 
    3: "qwen2.5:3b-instruct"
}

In [ ]:
# Load and concatenate all results into a single DataFrame
result_files = list(Path("../results/3m_200q_5r").glob("*.csv"))
print(f"Found {len(result_files)} result files:")
for f in result_files:
    print(f"  {f}")

df = pd.concat((pd.read_csv(f) for f in result_files), ignore_index=True)

Found 4 result files:
  ..\results\run_2\chain_20260314_045634.csv
  ..\results\run_2\full_20260313_224213.csv
  ..\results\run_2\independent_20260313_213131.csv
  ..\results\run_2\mediator_20260314_011147.csv


In [206]:
# Data overview
print("=== Dataset Summary ===")
print(f"Shape: {df.shape}")
print("\nTopology counts:")
print(df['topology'].value_counts())
print("\nRounds per topology:")
print(df.groupby('topology')['round'].max())
print("\nTemperature:        0.4 \nSamples questions:  200")
print(f"\nParse failure rate: {df['parse_failed'].mean():.2%}")

parse_by_model = df.groupby(["topology", "model"])["parse_failed"].mean().unstack()
print(parse_by_model.applymap(lambda x: f"{x:.1%}"))

=== Dataset Summary ===
Shape: (6104, 15)

Topology counts:
topology
chain          1904
full           1800
mediator       1800
independent     600
Name: count, dtype: int64

Rounds per topology:
topology
chain          4
full           3
independent    1
mediator       3
Name: round, dtype: int64

Temperature:        0.4 
Samples questions:  200

Parse failure rate: 8.40%
model       gemma3:4b llama3.2:3b phi4-mini qwen2.5:3b-instruct
topology                                                       
chain            2.7%       26.7%      1.5%                0.0%
full             3.3%       26.9%      0.4%                0.2%
independent      6.7%       37.3%      0.7%                0.0%
mediator         3.3%       31.6%      0.7%                0.0%


---
### 2. Accuracy Comparison Across Topologies

The fundamental question: **does collaboration help?**
We compare group-level accuracy (majority vote) across all topologies.

In [207]:
# Question-level accuracy table (one row per question per topology)
q_level = (
    df.groupby(["topology", "question_idx"])
    .agg(
        correct=("correct", "first"),
        expected=("expected_answer", "first"),
        group_answer=("group_answer", "first"),
        last_round_idx=("round", "idxmax"),
    )
    .reset_index()
)

last_round = df.loc[q_level["last_round_idx"]]

In [208]:
accuracy = q_level.groupby("topology")["correct"].mean().reset_index()
accuracy.columns = ["topology", "accuracy"]

In [209]:
fig = px.bar(
    accuracy,
    y="topology", x="accuracy",
    color="topology", color_discrete_map=TOPO_COLORS,
    text=accuracy["accuracy"].map("{:.1%}".format),
    title="Group Accuracy by Communication Topology",
    labels={"accuracy": "Accuracy", "topology": "Topology"},
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_range=[0, 1])

fig.show()
fig.write_image("figures/accuracy_by_topology.png", width=1500, scale=2)

#### Individual Agent Accuracy vs Group Accuracy

Does (majority) voting actually help? Comparing individual agent accuracy to the group's majority-vote accuracy.

In [210]:
# All rows with max round per group (one per agent)
last_round = df.loc[df["round"] == df.groupby(["topology", "question_idx"])["round"].transform("max")]

# Individual agent accuracy
individual_acc = (
    last_round.assign(correct=lambda d: d["answer"] == d["expected_answer"])
    .groupby(["topology", "agent_id"])["correct"].mean()
    .reset_index(name="accuracy")
)
individual_acc["agent_id"] = individual_acc["agent_id"].map(AGENT_NAME_MAP)

# Group accuracy
group_acc = (
    q_level.groupby("topology")["correct"].mean()
    .reset_index(name="accuracy")
    .assign(agent_id="Group Vote")
)

# Combining individual and group accuracy
combined = pd.concat([individual_acc, group_acc], ignore_index=True)
agent_color_map = {**AGENT_COLORS, "Group Vote": "#000000"}


fig = px.bar(
    combined, x="topology", y="accuracy",
    color="agent_id", barmode="group",
    title="Individual Agent vs Group Accuracy by Topology",
    labels={"accuracy": "Accuracy", "topology": "Topology", "agent_id": ""},
    text=combined["accuracy"].map("{:.1%}".format),
    category_orders={"topology": ["independent", "full", "mediator", "chain"]},
    color_discrete_map=agent_color_map
)

fig.update_traces(textposition="outside")
fig.update_layout(yaxis_range=[0, 1.1])

fig.show()
fig.write_image("figures/individual_vs_group_accuracy.png", scale=2, width=1500)

---
### 3. Confidence Analysis

Are agents overconfident? Does confidence actually predict correctness?

In [211]:
calibration = (
    last_round.groupby("topology")
    .agg(confidence=("confidence", "mean"), accuracy=("correct", lambda x: x.mean() * 100))
    .reset_index()
    .melt(id_vars="topology", var_name="Metric", value_name="value")
)

In [216]:
# Calibration Gap

# If the confidence bar is much taller than the accuracy bar, agents are **overconfident**
# A well-calibrated agent would have these roughly equal

calibration = (
    last_round.groupby("topology")
    .apply(lambda g: pd.Series({
        "Mean Confidence": g["confidence"].mean(),
        "Actual Accuracy (%)": (g["answer"] == g["expected_answer"]).mean() * 100,
    }))
    .reset_index()
)
calibration_melted = calibration.melt(
    id_vars="topology", var_name="Metric", value_name="value"
)

fig = px.bar(
    calibration_melted,
    x="topology", y="value",
    color="Metric", barmode="group",
    text=calibration_melted["value"].apply(lambda x: f"{x:.1f}%"),
    title="Calibration Gap: Mean Confidence vs Actual Accuracy",
    color_discrete_map={"Actual Accuracy (%)": "#2ecc71", "Mean Confidence": "#e74c3c"},
    labels={"value": "Percentage", "topology": "Topology"},
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_range=[0, 110])

fig.show()
fig.write_image("figures/calibration_gap.png", scale=2)

Agents report 90%+ confidence across all topologies regardless of correctness. Confidence is **not a reliable signal** in these small models.

## 4. Convergence Dynamics
For multi-round topologies (fully connected, mediator), do agents converge
toward agreement? And does that agreement move toward the **correct** answer?

In [213]:
multi_round = df[df["topology"].isin(["full", "mediator"])]

# Agreement per round
avg_agreement = (
    multi_round.groupby(["topology", "question_idx", "round"])["answer"]
    .apply(lambda a: (a.dropna() == a.dropna().mode().iloc[0]).mean() if len(a.dropna()) else 0)
    .reset_index(name="agreement")
    .groupby(["topology", "round"], as_index=False)["agreement"].mean()
)

# Group accuracy per round
avg_acc_round = (
    multi_round.groupby(["topology", "question_idx", "round"])
    .apply(lambda g: int(g["answer"].dropna().mode().iloc[0] == g["expected_answer"].iloc[0]) if len(g["answer"].dropna()) else 0)
    .reset_index(name="correct")
    .groupby(["topology", "round"], as_index=False)["correct"].mean()
)

In [215]:
# Subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Agent Agreement Rate Across Rounds", "Group Accuracy Across Rounds")
)

for topo in avg_agreement["topology"].unique():
    fig.add_trace(
        go.Scatter(
            x=avg_agreement[avg_agreement["topology"] == topo]["round"],
            y=avg_agreement[avg_agreement["topology"] == topo]["agreement"],
            name=topo, line=dict(color=TOPO_COLORS.get(topo))
        ),
        row=1, col=1
    )

for topo in avg_acc_round["topology"].unique():
    fig.add_trace(
        go.Scatter(
            x=avg_acc_round[avg_acc_round["topology"] == topo]["round"],
            y=avg_acc_round[avg_acc_round["topology"] == topo]["correct"],
            name=topo, line=dict(color=TOPO_COLORS.get(topo)),
            showlegend=False
        ),
        row=1, col=2
    )

fig.update_layout(
    yaxis_tickformat=".0%", yaxis_range=[0, 1.05],
    yaxis2_tickformat=".0%", yaxis2_range=[0, 1.05],
)

fig.show()
fig.write_image("figures/agreement_accuracy_across_rounds.png", scale=2)

Both topologies show strong convergence. Agreement rises from ~45% in round 1 to ~78% by round 3, with accuracy following the same upward trend (21% -> 80%). Confirming agents are converging toward correct answers, not just blindly agreeing. Notably, mediator reaches higher accuracy faster in round 2 despite slightly lower agreement than fully connected — suggesting the mediator's structured summary helps agents focus on the right answer more efficiently.